# Signal AUROC

AUROC scores (`dAUROC` and `AUROC_point_mass`) for each confidence signal (Linguistic Confidence, Token Probability, Semantic Uncertainty) across datasets and models. Values are mean ± std across models within each dataset.

In [10]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd

import sys
sys.path.insert(0, '/home/ivan/lm-confidence-evaluation-harness')

warnings.filterwarnings('ignore')

In [11]:
COMMON_DIR = '/hdd/ivny'
DATASETS = ['mmlu', 'squadv2', 'truthful_qa']
SIGNAL_ORDER = ['lc', 'tp', 'su']
SIGNALS = {
    'lc': 'Linguistic Confidence',
    'tp': 'Token Probability',
    'su': 'Semantic Uncertainty',
}
DATASET_LABELS = {
    'mmlu':        'MMLU',
    'squadv2':     'SQuAD 2.0',
    'truthful_qa': 'TruthfulQA',
}
METRICS = ['accuracy_scalar_with_abstention', 'AUROC_point_mass']
METRIC_LABELS = {
    'accuracy_scalar_with_abstention':           'accuracy',
    'dAUROC':           'dAUROC',
    'AUROC_point_mass': 'AUROC (mean)',
}


def _get_latest_leaf(model_path: str) -> str | None:
    try:
        subdirs = [
            os.path.join(model_path, d)
            for d in os.listdir(model_path)
            if os.path.isdir(os.path.join(model_path, d))
        ]
    except FileNotFoundError:
        return None
    if not subdirs:
        return None
    chosen = max(subdirs, key=os.path.getmtime)
    return chosen if os.path.exists(os.path.join(chosen, 'eval_metrics.csv')) else None

## Discover Runs

Walk `/hdd/ivny/results/{dataset}/direct_qa_{signal}/{org}/{model}/` and select the latest timestamp dir per model.

In [12]:
runs = []

for dataset in DATASETS:
    results_root = os.path.join(COMMON_DIR, 'results', dataset)
    try:
        dqa_dirs = sorted([d for d in os.listdir(results_root) if d.startswith('direct_qa_')])
    except FileNotFoundError:
        continue
    for dqa in dqa_dirs:
        suffix = dqa.replace('direct_qa_unified_', '').replace('direct_qa_', '')
        if suffix not in SIGNALS:
            continue
        dqa_path = os.path.join(results_root, dqa)
        for org in os.listdir(dqa_path):
            org_path = os.path.join(dqa_path, org)
            if not os.path.isdir(org_path):
                continue
            for model in os.listdir(org_path):
                if model in ["gpt-oss-120b", "Qwen3-235B-A22B-Instruct-2507-tput"]:
                    continue
                model_path = os.path.join(org_path, model)
                if not os.path.isdir(model_path):
                    continue
                leaf = _get_latest_leaf(model_path)
                if leaf is None:
                    continue
                runs.append({'dataset': dataset, 'signal': suffix, 'model': f'{org}/{model}', 'leaf': leaf})

print(f'Total runs found: {len(runs)}')
pd.DataFrame(runs)[['dataset', 'signal', 'model']].sort_values(['signal', 'dataset', 'model']).reset_index(drop=True)

Total runs found: 45


,dataset,signal,model
0,mmlu,lc,google/gemma-4-31B-it
1,mmlu,lc,meta-llama/Llama-3.1-8B-Instruct
2,mmlu,lc,mistralai/Mistral-7B-Instruct-v0.3
3,mmlu,lc,openai/gpt-oss-20b
4,mmlu,lc,qwen/Qwen3-8B
5,squadv2,lc,google/gemma-4-31B-it
6,squadv2,lc,meta-llama/Llama-3.1-8B-Instruct
7,squadv2,lc,mistralai/Mistral-7B-Instruct-v0.3
8,squadv2,lc,openai/gpt-oss-20b
9,squadv2,lc,qwen/Qwen3-8B


## Load Data

Read `eval_metrics.csv` for each run and average `dAUROC` and `AUROC_point_mass` across evaluation rounds.

In [13]:
records = []
for run in runs:
    csv_path = os.path.join(run['leaf'], 'eval_metrics.csv')
    try:
        df = pd.read_csv(csv_path)
    except Exception:
        continue
    if not all(m in df.columns for m in METRICS):
        continue
    row = {'dataset': run['dataset'], 'signal': run['signal'], 'model': run['model']}
    row.update(df[METRICS].mean().to_dict())
    records.append(row)

df_records = pd.DataFrame(records)
print(f'Loaded {len(df_records)} runs')
df_records.head()

Loaded 45 runs


,dataset,signal,model,accuracy_scalar_with_abstention,AUROC_point_mass
0,mmlu,lc,mistralai/Mistral-7B-Instruct-v0.3,0.570859,0.540209
1,mmlu,lc,google/gemma-4-31B-it,0.865404,0.506781
2,mmlu,lc,meta-llama/Llama-3.1-8B-Instruct,0.638869,0.555186
3,mmlu,lc,openai/gpt-oss-20b,0.805971,0.519726
4,mmlu,lc,qwen/Qwen3-8B,0.713075,0.569372


## Results Table

Rows: signals (LC, TP, SU). Columns: dataset × metric. Values: mean ± std across models.

In [14]:
agg = (
    df_records.groupby(['signal', 'dataset'])[METRICS]
    .agg(['mean', 'std'])
    .round(4)
    .reset_index()
)
agg.columns = ['signal', 'dataset'] + [f'{m}_{s}' for m in METRICS for s in ['mean', 'std']]

pivot_rows = []
for sig in SIGNAL_ORDER:
    row = {'Signal': SIGNALS[sig]}
    for ds in DATASETS:
        sub = agg[(agg['signal'] == sig) & (agg['dataset'] == ds)]
        for metric in METRICS:
            key = (DATASET_LABELS[ds], METRIC_LABELS[metric])
            if sub.empty:
                row[key] = '-'
            else:
                m = sub[f'{metric}_mean'].values[0]
                s = sub[f'{metric}_std'].values[0]
                row[key] = f'{m:.4f} ± {s:.4f}'
    pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows).set_index('Signal')
pivot_df.columns = pd.MultiIndex.from_tuples(pivot_df.columns)
pivot_df

MMLU                         SQuAD 2.0  \
                              accuracy     AUROC (mean)         accuracy   
Signal                                                                     
Linguistic Confidence  0.7188 ± 0.1198  0.5383 ± 0.0255  0.5641 ± 0.1003   
Token Probability      0.7190 ± 0.1197  0.6086 ± 0.0753  0.5644 ± 0.1003   
Semantic Uncertainty   0.7188 ± 0.1198  0.6815 ± 0.0929  0.5639 ± 0.1005   

                                             TruthfulQA                   
                          AUROC (mean)         accuracy     AUROC (mean)  
Signal                                                                    
Linguistic Confidence  0.4971 ± 0.0285  0.4926 ± 0.1027  0.6161 ± 0.0277  
Token Probability      0.6465 ± 0.0351  0.4955 ± 0.1007  0.6242 ± 0.0587  
Semantic Uncertainty   0.6471 ± 0.0698  0.4946 ± 0.1027  0.6423 ± 0.0175

## Mean Confidence by Signal and Dataset

Load `graded_outputs_0.pkl` per run, extract the mean of each `BetaDistribution` confidence (`mu`), and report the average confidence across all questions and models per (signal, dataset).

In [15]:
def _conf_to_float(c):
    """Extract scalar confidence from a BetaDistribution or plain float; returns None if invalid."""
    if c is None:
        return None
    if hasattr(c, 'mu'):
        return float(c.mu)
    try:
        return float(c)
    except (TypeError, ValueError):
        return None


conf_records = []
for run in runs:
    pkl_path = os.path.join(run['leaf'], 'graded_outputs_0.pkl')
    if not os.path.exists(pkl_path):
        continue
    try:
        with open(pkl_path, 'rb') as f:
            graded = pickle.load(f)
    except Exception:
        continue
    confs = [_conf_to_float(c) for c in graded.extracted_confidences[0]]
    confs = [c for c in confs if c is not None]
    if not confs:
        continue
    conf_records.append({
        'dataset': run['dataset'],
        'signal':  run['signal'],
        'model':   run['model'],
        'mean_conf': np.mean(confs),
    })

df_conf = pd.DataFrame(conf_records)

conf_agg = (
    df_conf.groupby(['signal', 'dataset'])['mean_conf']
    .agg(['mean', 'std'])
    .round(4)
    .reset_index()
)

conf_pivot_rows = []
for sig in SIGNAL_ORDER:
    row = {'Signal': SIGNALS[sig]}
    for ds in DATASETS:
        sub = conf_agg[(conf_agg['signal'] == sig) & (conf_agg['dataset'] == ds)]
        if sub.empty:
            row[DATASET_LABELS[ds]] = '-'
        else:
            m = sub['mean'].values[0]
            s = sub['std'].values[0]
            row[DATASET_LABELS[ds]] = f'{m:.4f} ± {s:.4f}'
    conf_pivot_rows.append(row)

conf_pivot_df = pd.DataFrame(conf_pivot_rows).set_index('Signal')
print('Mean confidence (µ of BetaDistribution) per signal and dataset (mean ± std across models):')
conf_pivot_df

Mean confidence (µ of BetaDistribution) per signal and dataset (mean ± std across models):


,MMLU,SQuAD 2.0,TruthfulQA
Signal,,,
Linguistic Confidence,0.7837 ± 0.0142,0.8404 ± 0.0147,0.8285 ± 0.0115
Token Probability,0.9358 ± 0.0445,0.8378 ± 0.1037,0.7041 ± 0.1651
Semantic Uncertainty,0.8907 ± 0.0539,0.8844 ± 0.0382,0.8080 ± 0.0542


## Per-Model Metrics

AUROC (mean) and accuracy per model, signal, and dataset.

In [17]:
MODEL_ORDER = sorted(df_records['model'].unique())

per_model_rows = []
for model in MODEL_ORDER:
    for sig in SIGNAL_ORDER:
        row = {'Model': model, 'Signal': SIGNALS[sig]}
        for ds in DATASETS:
            sub = df_records[(df_records['model'] == model) & (df_records['signal'] == sig) & (df_records['dataset'] == ds)]
            for metric in METRICS:
                key = (DATASET_LABELS[ds], METRIC_LABELS[metric])
                if sub.empty:
                    row[key] = '-'
                else:
                    row[key] = round(sub[metric].values[0], 4)
        per_model_rows.append(row)

per_model_df = pd.DataFrame(per_model_rows).set_index(['Model', 'Signal'])
per_model_df.columns = pd.MultiIndex.from_tuples(per_model_df.columns)
per_model_df.round(3)

MMLU  \
                                                         accuracy   
Model                              Signal                           
google/gemma-4-31B-it              Linguistic Confidence    0.865   
                                   Token Probability        0.865   
                                   Semantic Uncertainty     0.865   
meta-llama/Llama-3.1-8B-Instruct   Linguistic Confidence    0.639   
                                   Token Probability        0.639   
                                   Semantic Uncertainty     0.639   
mistralai/Mistral-7B-Instruct-v0.3 Linguistic Confidence    0.571   
                                   Token Probability        0.571   
                                   Semantic Uncertainty     0.571   
openai/gpt-oss-20b                 Linguistic Confidence    0.806   
                                   Token Probability        0.806   
                                   Semantic Uncertainty     0.806   
qwen/Qwen3-8B                      Linguistic Confidence    0.713   
                                   Token Probability        0.713   
                                   Semantic Uncertainty     0.713   

                                                                       \
                                                         AUROC (mean)   
Model                              Signal                               
google/gemma-4-31B-it              Linguistic Confidence        0.507   
                                   Token Probability            0.637   
                                   Semantic Uncertainty         0.613   
meta-llama/Llama-3.1-8B-Instruct   Linguistic Confidence        0.555   
                                   Token Probability            0.688   
                                   Semantic Uncertainty         0.762   
mistralai/Mistral-7B-Instruct-v0.3 Linguistic Confidence        0.540   
                                   Token Probability            0.657   
                                   Semantic Uncertainty         0.640   
openai/gpt-oss-20b                 Linguistic Confidence        0.520   
                                   Token Probability            0.556   
                                   Semantic Uncertainty         0.799   
qwen/Qwen3-8B                      Linguistic Confidence        0.569   
                                   Token Probability            0.506   
                                   Semantic Uncertainty         0.594   

                                                         SQuAD 2.0  \
                                                          accuracy   
Model                              Signal                            
google/gemma-4-31B-it              Linguistic Confidence     0.738   
                                   Token Probability         0.738   
                                   Semantic Uncertainty      0.738   
meta-llama/Llama-3.1-8B-Instruct   Linguistic Confidence     0.556   
                                   Token Probability         0.558   
                                   Semantic Uncertainty      0.558   
mistralai/Mistral-7B-Instruct-v0.3 Linguistic Confidence     0.529   
                                   Token Probability         0.527   
                                   Semantic Uncertainty      0.528   
openai/gpt-oss-20b                 Linguistic Confidence     0.494   
                                   Token Probability         0.495   
                                   Semantic Uncertainty      0.494   
qwen/Qwen3-8B                      Linguistic Confidence     0.503   
                                   Token Probability         0.503   
                                   Semantic Uncertainty      0.502   

                                                                       \
                                                         AUROC (mean)   
Model                              Signal                               
google/gemma-4-31B-it  